
# 📘 08_Dashboard_Rewriter — Update AI/BI Dashboard References

This notebook automatically rewrites the **AI/BI dashboard JSON** to match your current environment configuration.

It replaces hardcoded catalog/schema names with your selected values, so the same dashboard template can be reused across multiple Databricks workspaces or catalogs.

---

### 📂 Folder Structure

📁 notebooks/

    └── 08_Dashboard_Rewriter.ipynb
📁 dashboards/

    └── Data Quality Dashboards Template.lvdash.json   ← input template
    └── Data Quality Dashboards.lvdash.json            ← rewritten output

---

### ⚙️ What It Does

- Reads the dashboard JSON template  
  `/../../dashboards/Data Quality Dashboards Template.lvdash.json`
- Rewrites all SQL references (`FROM catalog.schema.table`) → `{catalog}.{out_schema}.{view_name}`
- Writes a new `.lvdash.json` ready for import into **AI/BI Dashboard**

In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog",     "dbdemos_steventan",                 "Catalog")
dbutils.widgets.text("out_schema",  "lakehouse_monitoring_demo_results", "Output Schema (results)")
dbutils.widgets.text("data_schema", "lakehouse_monitoring",              "Data Schema (raw tables)")

catalog     = dbutils.widgets.get("catalog").strip()
out_schema  = dbutils.widgets.get("out_schema").strip()
data_schema = dbutils.widgets.get("data_schema").strip()

print(f"📦 Catalog: {catalog}")
print(f"📦 Output Schema: {out_schema}")
print(f"📦 Data Schema: {data_schema}")

# --- resolve relative paths (assuming folder structure: project/{notebooks,dashboards}) ---
from pyspark.dbutils import DBUtils
import os

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_notebooks_dir = os.path.dirname(_nb_path)

if _notebooks_dir.endswith("/notebooks"):
    _project_root = _notebooks_dir[:-len("/notebooks")]
else:
    _project_root = os.path.dirname(_notebooks_dir)

dash_dir_ws = f"{_project_root}/dashboards"

template_ws = f"{dash_dir_ws}/Data Quality Dashboards Template.lvdash.json"
output_ws   = f"{dash_dir_ws}/Data Quality Dashboards.lvdash.json"

def ws_to_local(path_ws: str) -> str:
    if path_ws.startswith("/Workspace/"): return path_ws
    if path_ws.startswith("/"): return "/Workspace" + path_ws
    return "/Workspace/" + path_ws

template_local = ws_to_local(template_ws)
output_local   = ws_to_local(output_ws)

print(f"🗂 Template: {template_local}")
print(f"🗂 Output:   {output_local}")

In [0]:
# Databricks notebook source
# Read template file and replace old identifiers with widget values
import os

with open(template_local, "r", encoding="utf-8") as f:
    content = f.read()

# Define replacement mapping
REPLACEMENTS = {
    "dbdemos_steventan": catalog,
    "lakehouse_monitoring_demo_results": out_schema,
    "lakehouse_monitoring": data_schema,
}

# Perform replacements
count = 0
for old, new in REPLACEMENTS.items():
    c = content.count(old)
    if c:
        content = content.replace(old, new)
        count += c
        print(f"🔄 Replaced '{old}' → '{new}' ({c} occurrences)")

# Write the new dashboard file
os.makedirs(os.path.dirname(output_local), exist_ok=True)
with open(output_local, "w", encoding="utf-8") as f:
    f.write(content)

print(f"\n✅ Dashboard JSON updated successfully: {output_local}")
print(f"🧾 Total replacements: {count}")